<a href="https://colab.research.google.com/github/f171p/323project/blob/main/HighScoreList.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from typing import List, Tuple, Optional
import csv
import os
from functools import reduce

# HighScore-Typ als Tuple: Spielername, Datum, Level, Zeit in Sekunden
HighScore = Tuple[str, str, str, int]
HighScoreDict = dict[str, List[HighScore]]  # Dictionary: Level -> Liste von HighScores

# Pfad zur CSV-Datei
FILE_PATH = "sudoku_highscores.csv"

In [ ]:
# Pure Function – gibt leere Struktur für Highscores zurück
def leeres_score_dict() -> HighScoreDict:
    return {"einfach": [], "mittel": [], "schwer": [], "genie": []}

# Pure Function – liest eine Zeile aus der CSV und gibt ein HighScore-Tuple zurück
def parse_row(row: dict) -> HighScore:
    return (row["Spielername"], row["Datum"], row["Level"].lower(), int(row["Benötigte Zeit (Sekunden)"]))

# Pure Function – fügt neuen Score ins richtige Level ein, sortiert und kürzt auf Top 10
# → Keine Mutation! Gibt neue Kopie des Dictionaries zurück → Immutability
def insert_score(scores: HighScoreDict, score: HighScore) -> HighScoreDict:
    level = score[2]
    neue_liste = sorted(scores.get(level, []) + [score], key=lambda x: x[3])[:10]
    return {**scores, level: neue_liste}  # Rückgabe neuer Dict → kein Side-Effect

# Pure Function – berechnet Rang, falls Score in Top 10 passt
def get_rank(scores: HighScoreDict, level: str, time: int) -> Optional[int]:
    level_scores = scores.get(level, [])
    if len(level_scores) < 10 or time < level_scores[-1][3]:
        return next((i + 1 for i, (_, _, _, t) in enumerate(level_scores) if time < t), len(level_scores) + 1)
    return None

# Pure Function – gibt eine formatierte String-Repräsentation der Highscores zurück
def display_highscores(scores: HighScoreDict, level: str) -> str:
    return "\n".join(f"{i + 1}. {name} - {date} - {time}s"
                     for i, (name, date, _, time) in enumerate(scores.get(level, [])))

# Pure Function – gibt neuen Zustand zurück, ggf. mit neuem Score eingetragen
def add_score(scores: HighScoreDict, name: str, date: str, level: str, time: int) -> Tuple[str, HighScoreDict]:
    rank = get_rank(scores, level, time)
    if rank:
        neue_scores = insert_score(scores, (name, date, level, time))
        return f"Your rank: {rank}!", neue_scores
    return f"HighScore entries only better than {scores[level][-1][3]} seconds", scores

# Ladevorgang – aus Datei lesen → hier bewusst kein pure function, da IO-Operation
def load_scores(file_path: str) -> HighScoreDict:
    if not os.path.exists(file_path):
        return leeres_score_dict()
    with open(file_path, newline='', encoding='ISO-8859-1') as csvfile:
        reader = csv.DictReader(csvfile, delimiter=',')
        data = list(map(parse_row, reader))  # Map-Funktion
        return reduce(insert_score, data, leeres_score_dict())  # reduce = rekursionsähnlich

# Speichern der Daten – IO-Operation (nicht pure), aber zentral
def save_scores(file_path: str, scores: HighScoreDict):
    with open(file_path, mode='w', newline='', encoding='ISO-8859-1') as csvfile:
        fieldnames = ["Spielername", "Datum", "Level", "Benötigte Zeit (Sekunden)"]
        writer = csv.DictWriter(csvfile, fieldnames=fieldnames, delimiter=',')
        writer.writeheader()
        for level, level_scores in scores.items():
            for score in level_scores:
                writer.writerow({"Spielername": score[0], "Datum": score[1], "Level": score[2], "Benötigte Zeit (Sekunden)": score[3]})

# Einzige Funktion mit Konsolen-Ausgabe – Trennung von UI und Logik (Modulziel!)
def menu(actions: List[Tuple], file_path: str):
    scores = load_scores(file_path)  # Initiale Ladeaktion

    for action in actions:
        match action:  # Pattern Matching – statt if-else
            case ("load",):
                scores = load_scores(file_path)
                print("Highscores geladen.")
            case ("display", level):
                output = display_highscores(scores, level)
                print(output)
            case ("add", name, date, level, time):
                message, scores = add_score(scores, name, date, level, time)
                print(message)
            case ("save",):
                save_scores(file_path, scores)
                print("Highscores gespeichert.")

In [ ]:
test_actions = [
    ("load",),
    ("add", "Lina", "2025-04-13", "genie", 75),
    ("display", "genie"),
    ("save",)
]

# Startpunkt der Anwendung – "main"
menu(test_actions, FILE_PATH)

Highscores geladen.
Your rank: 1!
1. Lina - 2025-04-13 - 75s
2. Artorias - 30/03/2025 - 1241s
3. Samuel - 31/03/2025 - 1477s
4. Helena - 01/04/2025 - 1496s
5. Clara - 02/04/2025 - 1570s
Highscores gespeichert.
